- This notebook queries the Wikidata SPARQL edpoint to download details about the UEFA football players using filter:

>{?player p:P2276 ?UEFA_id} #to limit the data size filter only UEFA players

> Name, DOB, Age, Sex, PlayedFor, TransferMarketID, starttime, endtime

In [139]:
from SPARQLWrapper import SPARQLWrapper, JSON
from rdflib import Graph
import datetime
import pandas as pd

In [171]:
## Query Construction
"""
Player data Name, BOB, Age, Sex, PlayedFor, TransferMarketID, starttime, endtime
"""
#endpoint
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)

player_data_query = """


PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
PREFIX ps: <http://www.wikidata.org/prop/statement/>
PREFIX p: <http://www.wikidata.org/prop/>
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>

SELECT ?fullName ?player ?DOB ?sex ?transferMarketID ?team  ?startTime ?endTime  ?teamCountry
WHERE {
   ?player wdt:P106 wd:Q937857; # Football association player
            p:P569 ?DOB_statement;
            p:P21 ?sex_statement;
            p:P2446 ?transferMarket_statement;
            p:P54 ?team_statement ;
            p:P1559 ?fullName_statement.
  
   ?fullName_statement ps:P1559 ?fullName.         
   OPTIONAL{?DOB_statement ps:P569 ?DOB}.
   OPTIONAL{?sex_statement ps:P21 ?sex}.
   OPTIONAL{?transferMarket_statement  ps:P2446 ?transferMarketID}.
   
   ?team_statement ps:P54 ?team ;
   OPTIONAL{?team_statement pq:P580 ?startTime} .
   OPTIONAL{?team_statement pq:P582 ?endTime} .
                   
   ?team p:P17 ?country_statement.
   ?country_statement ps:P17 ?teamCountry.
   
   #VALUES ?player {wd:Q615}.
   # VALUES ?relation {p:P54}.
   # https://www.wikidata.org/wiki/Property:P569 #DOB
   # https://www.wikidata.org/wiki/Property:P21  #Sex
   # https://www.wikidata.org/wiki/Property:P735 #Given Name
   # https://www.wikidata.org/wiki/Property:P734 #Family Name
   # https://www.wikidata.org/wiki/Property:P118 #League
   # https://www.wikidata.org/wiki/Property:P17  #Country
   # https://www.wikidata.org/wiki/Property:P2446 #transferMarketID
   # https://www.transfermarkt.com/transfermarkt/profil/spieler/  #transfermarket URL
  
  FILTER EXISTS {?player p:P2276 ?UEFA_id} #to limit the data size filter only UEFA players
  
  }
  
"""

sparql.setQuery(player_data_query)


player_results = sparql.query().convert()

In [141]:
print(type(player_results))

<class 'dict'>


In [142]:
#Sample
player_results["results"]["bindings"][0]

{'transferMarketID': {'type': 'literal', 'value': '57079'},
 'player': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q296833'},
 'fullName': {'xml:lang': 'en', 'type': 'literal', 'value': 'Jack Rodwell'},
 'team': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q5794'},
 'DOB': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime',
  'type': 'literal',
  'value': '1991-03-11T00:00:00Z'},
 'startTime': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime',
  'type': 'literal',
  'value': '2007-01-01T00:00:00Z'},
 'sex': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q6581097'},
 'teamCountry': {'type': 'uri',
  'value': 'http://www.wikidata.org/entity/Q145'},
 'endTime': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime',
  'type': 'literal',
  'value': '2012-01-01T00:00:00Z'}}

In [172]:
#Total records
len(player_results["results"]["bindings"])

27446

In [173]:
#column list
cols = list(player_results["results"]["bindings"][1].keys())
cols

['player',
 'fullName',
 'team',
 'teamCountry',
 'DOB',
 'sex',
 'transferMarketID',
 'startTime',
 'endTime']

In [174]:
#creating a data dictionary
data = {key:[] for key in cols}
data

{'player': [],
 'fullName': [],
 'team': [],
 'teamCountry': [],
 'DOB': [],
 'sex': [],
 'transferMarketID': [],
 'startTime': [],
 'endTime': []}

In [175]:
#ignore
for i in data.keys():
    print('data["'+i+'"].append(result["'+i+'"]["value"])')

data["player"].append(result["player"]["value"])
data["fullName"].append(result["fullName"]["value"])
data["team"].append(result["team"]["value"])
data["teamCountry"].append(result["teamCountry"]["value"])
data["DOB"].append(result["DOB"]["value"])
data["sex"].append(result["sex"]["value"])
data["transferMarketID"].append(result["transferMarketID"]["value"])
data["startTime"].append(result["startTime"]["value"])
data["endTime"].append(result["endTime"]["value"])


In [176]:
for item in player_results["results"]["bindings"]:
    for col in cols:
        if col in item:
            data[col].append(item[col]['value'])
        else:
            data[col].append("")

In [177]:
#create a dataframe out of the data dictioanry and save it to a csv
data_frame = pd.DataFrame(data)
print('There are',len(data_frame),'records in the dataframe')
data_frame.head()

There are 27446 records in the dataframe


,player,fullName,team,teamCountry,DOB,sex,transferMarketID,startTime,endTime
0,http://www.wikidata.org/entity/Q730284,Gökhan Gönül,http://www.wikidata.org/entity/Q28380902,http://www.wikidata.org/entity/Q43,1985-01-04T00:00:00Z,http://www.wikidata.org/entity/Q6581097,33793,2016-01-01T00:00:00Z,2020-01-01T00:00:00Z
1,http://www.wikidata.org/entity/Q730284,Gökhan Gönül,http://www.wikidata.org/entity/Q28380902,http://www.wikidata.org/entity/Q43,1985-01-07T00:00:00Z,http://www.wikidata.org/entity/Q6581097,33793,2016-01-01T00:00:00Z,2020-01-01T00:00:00Z
2,http://www.wikidata.org/entity/Q730284,Gökhan Gönül,http://www.wikidata.org/entity/Q641373,http://www.wikidata.org/entity/Q43,1985-01-04T00:00:00Z,http://www.wikidata.org/entity/Q6581097,33793,2003-01-01T00:00:00Z,2004-01-01T00:00:00Z
3,http://www.wikidata.org/entity/Q730284,Gökhan Gönül,http://www.wikidata.org/entity/Q641373,http://www.wikidata.org/entity/Q43,1985-01-07T00:00:00Z,http://www.wikidata.org/entity/Q6581097,33793,2003-01-01T00:00:00Z,2004-01-01T00:00:00Z
4,http://www.wikidata.org/entity/Q730284,Gökhan Gönül,http://www.wikidata.org/entity/Q340133,http://www.wikidata.org/entity/Q43,1985-01-04T00:00:00Z,http://www.wikidata.org/entity/Q6581097,33793,2006-01-01T00:00:00Z,2006-01-01T00:00:00Z


In [178]:
data_frame.columns

Index(['player', 'fullName', 'team', 'teamCountry', 'DOB', 'sex',
       'transferMarketID', 'startTime', 'endTime'],
      dtype='object')

In [179]:
#Total possible edges excluding temporal metadata
data_frame[['transferMarketID', 'player', 'fullName', 'team', 'DOB', 'sex', 'teamCountry']].size

192122

In [181]:
#data_frame.to_csv('./Data/playerRawData.csv', index=False)